In [0]:
df = spark.read.format("delta").load("/delta/silver/transactions_with_customers")

# Apply AQE(Adaptive Query Execution) for automated optimization
spark.conf.set("spark.sql.adaptive.enabled","true")

# Optimizes the skew Join
spark.conf.set("spark.sql.adaptive.skewJoin.enabled","true")

"""
1. Manual optimization for shuffle tuning (Optional)

spark.conf.set("spark.sql.shuffle.partitions","50")

Manual number is calculated based on the datasize
Example: lIf the data size is 10GB which is 10,000MB, One Partition (a Single Task can handle 100 - 300MB of data efficiently).

So, 10000 MB / 200 MB = 50 Partitions.

2. Handling Data Skewness using Salting Technnique (Optional)

from pyspark.sql.functions import rand, floor

df_salted = df.withColumn(
    "salt",
    floor(rand()) * 10) # 10 buckets
    )

# Create Composite Key
df_salted = df_salted.withColumn(
    "user_salted",
    F.concat(F.col("user_id), F.lit("_"), F.col("salt"))
    )

df_salted = df_salted.repartition("user_salted")

"""

df = df.repartition("user_id") # Reduces Shuffle while performing groupBy
# but here there could be data skewness


gold = df.groupBy("user_id")\
        .agg(
            F.count(transaction_id).alias("transaction_count"),
            F.sum(amount).alias("total_spent"),
            F.avg(amount).alias("avg_spent")
        )
gold = gold.withColumn(
    "fraud_flag",
    F.when(
        (F.col("transaction_count") > 100) & (F.col("avg_spent") > 500), 1).otherwise(0)
    )

gold.write.format("delta").mode("overwrite").save("/delta/gold/fraud")

Add Fraud Rules

In [0]:
# Rule 1 - High Amount

df = df.withColumn(
    "high_amount_flag",
    when(
        col("amount") > 1000000,
        1
    ).otherwise(0)
)

# Rule 2

df = df.withColumns(
    "night_flag",
    when(
        hour(
            col("transaction_timestamp")
        ).between(0,4),
        1
    ).otherwise(0)
)

# Rule 3

df = df.withColumn(
    "geo_flag",
    when(
        col("city") != ("transaction_city"),1
    ).otherwise(0)
)

# Rule 4

velocity = (df.groupBy(
    window("transaction_timestamp","2 minutes"), "customer_id"
).count())

velocity_df = velocity.withColumn(
    "velocity_flag",
    when(col("count") > 5, 1).otherwise(0)
)

# Final score

df = df.join(
    velocity_df.select("customer_id","velocity_flag"),"customer_id","left"
)


df = df.fillna({"velocity_flag":0})

df = df.withColumn(
    "fraud_score",
    col("high_amount_flag") + ("night_flag") + ("geo_flag") + col("velocity_flag")
)

df = df.withColumn(
    "fraud_status",
    when(
        col("fraud_score") >= 2,
        "SUSPECTED_FRAUD"
    ).otherwise("NORMAL")
)

df.writeStream.format("delta").partitionBy("transaction_date")
        .option("checkpointLocation","/chk/gold").outputMode("append").table("gold.fraud_transactions")